In [ ]:
<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<title>해남군 전체 농경지 일괄 추출</title>
<script src="https://agis.epis.or.kr/ASD/pub2/js/jquery-3.4.1.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/proj4js/2.9.0/proj4.js"></script>
<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: 'Malgun Gothic', sans-serif; background: #f5f6f8; color: #1a1a1a; padding: 20px; }
.header { background: #244b6d; color: white; padding: 20px; border-radius: 8px; margin-bottom: 20px; }
.header h1 { font-size: 22px; margin-bottom: 5px; }
.header p { opacity: 0.85; font-size: 13px; }
.panel { background: white; border-radius: 8px; padding: 20px; margin-bottom: 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }
.panel h2 { font-size: 15px; color: #244b6d; margin-bottom: 12px; border-bottom: 2px solid #244b6d; padding-bottom: 6px; }
.btn-row { display: flex; gap: 10px; flex-wrap: wrap; margin-bottom: 10px; }
button { padding: 10px 18px; border: none; border-radius: 5px; cursor: pointer; font-weight: bold; font-size: 14px; transition: all 0.15s; }
.btn-primary { background: #e67e22; color: white; }
.btn-primary:hover { background: #d35400; }
.btn-stop { background: #c0392b; color: white; }
.btn-stop:hover { background: #a93226; }
.btn-success { background: #27ae60; color: white; }
.btn-success:hover { background: #1e8449; }
.btn-secondary { background: #95a5a6; color: white; }
button:disabled { opacity: 0.4; cursor: not-allowed; }
.stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 12px; }
.stat-box { background: #ecf0f1; padding: 12px; border-radius: 6px; text-align: center; }
.stat-box .label { font-size: 11px; color: #7f8c8d; text-transform: uppercase; }
.stat-box .value { font-size: 22px; font-weight: bold; color: #244b6d; margin-top: 4px; }
.progress-bar { background: #ecf0f1; border-radius: 12px; height: 24px; overflow: hidden; position: relative; }
.progress-fill { background: linear-gradient(90deg, #27ae60, #2ecc71); height: 100%; transition: width 0.3s; }
.progress-text { position: absolute; top: 0; left: 0; right: 0; bottom: 0; display: flex; align-items: center; justify-content: center; font-size: 12px; font-weight: bold; color: #1a1a1a; }
#log { background: #1a1a1a; color: #2ecc71; padding: 15px; border-radius: 5px; font-family: Consolas, Monaco, monospace; font-size: 12px; height: 280px; overflow-y: scroll; line-height: 1.6; }
#log .error { color: #e74c3c; }
#log .warn { color: #f39c12; }
#log .info { color: #3498db; }
.options { display: flex; gap: 20px; align-items: center; flex-wrap: wrap; }
.options label { font-size: 14px; cursor: pointer; }
.options input[type="checkbox"] { margin-right: 5px; }
.options input[type="number"] { width: 70px; padding: 5px; border: 1px solid #ccc; border-radius: 4px; }
.summary-table { width: 100%; border-collapse: collapse; font-size: 13px; }
.summary-table th { background: #244b6d; color: white; padding: 8px; text-align: left; }
.summary-table td { padding: 6px 8px; border-bottom: 1px solid #ecf0f1; }
.summary-table tr:hover { background: #f8f9fa; }
</style>
</head>
<body>

<div class="header">
  <h1>🌾 전라남도 해남군 농경지 팜맵 일괄 추출</h1>
  <p>178개 리 × 5개 농경지 분류 = 최대 890건 API 호출 · 결과를 CSV/GeoJSON으로 저장</p>
</div>

<div class="panel">
  <h2>실행 옵션</h2>
  <div class="options">
    <label><input type="checkbox" id="opt-01" checked> 01 논</label>
    <label><input type="checkbox" id="opt-02" checked> 02 밭</label>
    <label><input type="checkbox" id="opt-03" checked> 03 과수</label>
    <label><input type="checkbox" id="opt-04" checked> 04 시설</label>
    <label><input type="checkbox" id="opt-06"> 06 비경지 (양이 매우 많음)</label>
    <label>호출 간격: <input type="number" id="opt-delay" value="150" min="50" max="2000"> ms</label>
  </div>
</div>

<div class="panel">
  <h2>진행 상황</h2>
  <div class="stats">
    <div class="stat-box"><div class="label">진행</div><div class="value"><span id="stat-done">0</span>/<span id="stat-total">0</span></div></div>
    <div class="stat-box"><div class="label">수집 필지</div><div class="value" id="stat-parcels">0</div></div>
    <div class="stat-box"><div class="label">실패</div><div class="value" id="stat-fail" style="color:#c0392b;">0</div></div>
    <div class="stat-box"><div class="label">소요</div><div class="value" id="stat-time">-</div></div>
  </div>
  <div class="progress-bar">
    <div class="progress-fill" id="progress-fill" style="width:0%"></div>
    <div class="progress-text" id="progress-text">대기 중</div>
  </div>
</div>

<div class="panel">
  <h2>실행 / 다운로드</h2>
  <div class="btn-row">
    <button class="btn-primary" id="btn-start">▶ 추출 시작</button>
    <button class="btn-stop" id="btn-stop" disabled>■ 중지</button>
    <button class="btn-success" id="btn-csv" disabled>📊 CSV 다운로드</button>
    <button class="btn-success" id="btn-geojson" disabled>🗺️ GeoJSON 다운로드</button>
    <button class="btn-secondary" id="btn-clear">🗑️ 초기화</button>
  </div>
</div>

<div class="panel">
  <h2>실시간 로그</h2>
  <div id="log"></div>
</div>

<div class="panel" id="summary-panel" style="display:none;">
  <h2>읍면별 수집 통계</h2>
  <table class="summary-table" id="summary-table"></table>
</div>

<script>
// ============================================================
// 1. 설정
// ============================================================
const API_KEY = "IDby3c0TrxLEDmodNRVi";
const DOMAIN = "http://112.140.157.225";
const API_URL = "https://agis.epis.or.kr/ASD/farmmapApi/getFarmmapDataSeachBjdAndLandCode.do";

// 해남군 178개 리 코드
const CODES = [
  {c:"4682025021",n:"해남읍 해리"},
  {c:"4682025022",n:"해남읍 평동리"},
  {c:"4682025023",n:"해남읍 읍내리"},
  {c:"4682025024",n:"해남읍 고도리"},
  {c:"4682025025",n:"해남읍 남외리"},
  {c:"4682025026",n:"해남읍 성내리"},
  {c:"4682025027",n:"해남읍 수성리"},
  {c:"4682025028",n:"해남읍 신안리"},
  {c:"4682025029",n:"해남읍 연동리"},
  {c:"4682025030",n:"해남읍 안동리"},
  {c:"4682025031",n:"해남읍 백야리"},
  {c:"4682025032",n:"해남읍 내사리"},
  {c:"4682025033",n:"해남읍 남동리"},
  {c:"4682025034",n:"해남읍 남천리"},
  {c:"4682025035",n:"해남읍 용정리"},
  {c:"4682025036",n:"해남읍 구교리"},
  {c:"4682025037",n:"해남읍 복평리"},
  {c:"4682025038",n:"해남읍 부호리"},
  {c:"4682031021",n:"삼산면 신흥리"},
  {c:"4682031022",n:"삼산면 창리"},
  {c:"4682031023",n:"삼산면 송정리"},
  {c:"4682031024",n:"삼산면 봉학리"},
  {c:"4682031025",n:"삼산면 원진리"},
  {c:"4682031026",n:"삼산면 충리"},
  {c:"4682031027",n:"삼산면 구림리"},
  {c:"4682031028",n:"삼산면 평활리"},
  {c:"4682031029",n:"삼산면 상가리"},
  {c:"4682032021",n:"화산면 해창리"},
  {c:"4682032022",n:"화산면 금풍리"},
  {c:"4682032023",n:"화산면 연곡리"},
  {c:"4682032024",n:"화산면 율동리"},
  {c:"4682032025",n:"화산면 가좌리"},
  {c:"4682032026",n:"화산면 관동리"},
  {c:"4682032027",n:"화산면 월호리"},
  {c:"4682032028",n:"화산면 연정리"},
  {c:"4682032029",n:"화산면 방축리"},
  {c:"4682032030",n:"화산면 부길리"},
  {c:"4682032031",n:"화산면 송산리"},
  {c:"4682032032",n:"화산면 석호리"},
  {c:"4682032033",n:"화산면 안호리"},
  {c:"4682032034",n:"화산면 평호리"},
  {c:"4682032035",n:"화산면 삼마리"},
  {c:"4682033021",n:"현산면 만안리"},
  {c:"4682033022",n:"현산면 구시리"},
  {c:"4682033023",n:"현산면 고현리"},
  {c:"4682033024",n:"현산면 덕흥리"},
  {c:"4682033025",n:"현산면 일평리"},
  {c:"4682033026",n:"현산면 읍호리"},
  {c:"4682033027",n:"현산면 백포리"},
  {c:"4682033028",n:"현산면 초호리"},
  {c:"4682033029",n:"현산면 황산리"},
  {c:"4682033030",n:"현산면 구산리"},
  {c:"4682033031",n:"현산면 조산리"},
  {c:"4682033032",n:"현산면 월송리"},
  {c:"4682034021",n:"송지면 금강리"},
  {c:"4682034022",n:"송지면 군곡리"},
  {c:"4682034023",n:"송지면 가차리"},
  {c:"4682034024",n:"송지면 학가리"},
  {c:"4682034025",n:"송지면 우근리"},
  {c:"4682034026",n:"송지면 미야리"},
  {c:"4682034027",n:"송지면 산정리"},
  {c:"4682034028",n:"송지면 어란리"},
  {c:"4682034029",n:"송지면 소죽리"},
  {c:"4682034030",n:"송지면 송호리"},
  {c:"4682034031",n:"송지면 통호리"},
  {c:"4682034032",n:"송지면 마봉리"},
  {c:"4682034033",n:"송지면 해원리"},
  {c:"4682034034",n:"송지면 서정리"},
  {c:"4682035021",n:"북평면 남창리"},
  {c:"4682035022",n:"북평면 이진리"},
  {c:"4682035023",n:"북평면 서홍리"},
  {c:"4682035024",n:"북평면 평암리"},
  {c:"4682035025",n:"북평면 영전리"},
  {c:"4682035026",n:"북평면 오산리"},
  {c:"4682035027",n:"북평면 동해리"},
  {c:"4682035028",n:"북평면 와룡리"},
  {c:"4682036021",n:"북일면 만수리"},
  {c:"4682036022",n:"북일면 금당리"},
  {c:"4682036023",n:"북일면 신월리"},
  {c:"4682036024",n:"북일면 흥촌리"},
  {c:"4682036025",n:"북일면 운전리"},
  {c:"4682036026",n:"북일면 용일리"},
  {c:"4682036027",n:"북일면 방산리"},
  {c:"4682036028",n:"북일면 내동리"},
  {c:"4682037021",n:"옥천면 영춘리"},
  {c:"4682037022",n:"옥천면 영신리"},
  {c:"4682037023",n:"옥천면 신계리"},
  {c:"4682037024",n:"옥천면 신죽리"},
  {c:"4682037025",n:"옥천면 팔산리"},
  {c:"4682037026",n:"옥천면 월평리"},
  {c:"4682037027",n:"옥천면 용산리"},
  {c:"4682037028",n:"옥천면 성산리"},
  {c:"4682037029",n:"옥천면 흑천리"},
  {c:"4682037030",n:"옥천면 청신리"},
  {c:"4682037031",n:"옥천면 대산리"},
  {c:"4682037032",n:"옥천면 백호리"},
  {c:"4682037033",n:"옥천면 송산리"},
  {c:"4682037034",n:"옥천면 용동리"},
  {c:"4682038021",n:"계곡면 성진리"},
  {c:"4682038022",n:"계곡면 법곡리"},
  {c:"4682038023",n:"계곡면 강절리"},
  {c:"4682038024",n:"계곡면 당산리"},
  {c:"4682038025",n:"계곡면 장소리"},
  {c:"4682038026",n:"계곡면 선진리"},
  {c:"4682038027",n:"계곡면 반계리"},
  {c:"4682038028",n:"계곡면 방춘리"},
  {c:"4682038029",n:"계곡면 덕정리"},
  {c:"4682038030",n:"계곡면 여수리"},
  {c:"4682038031",n:"계곡면 사정리"},
  {c:"4682038032",n:"계곡면 가학리"},
  {c:"4682038033",n:"계곡면 잠두리"},
  {c:"4682038034",n:"계곡면 신평리"},
  {c:"4682038035",n:"계곡면 황죽리"},
  {c:"4682039021",n:"마산면 화내리"},
  {c:"4682039022",n:"마산면 장촌리"},
  {c:"4682039023",n:"마산면 맹진리"},
  {c:"4682039024",n:"마산면 송석리"},
  {c:"4682039025",n:"마산면 외호리"},
  {c:"4682039026",n:"마산면 산막리"},
  {c:"4682039027",n:"마산면 노하리"},
  {c:"4682039028",n:"마산면 연구리"},
  {c:"4682039029",n:"마산면 학의리"},
  {c:"4682039030",n:"마산면 용전리"},
  {c:"4682039031",n:"마산면 상등리"},
  {c:"4682040021",n:"황산면 일신리"},
  {c:"4682040022",n:"황산면 원호리"},
  {c:"4682040023",n:"황산면 연호리"},
  {c:"4682040024",n:"황산면 송호리"},
  {c:"4682040025",n:"황산면 우항리"},
  {c:"4682040026",n:"황산면 호동리"},
  {c:"4682040027",n:"황산면 한자리"},
  {c:"4682040028",n:"황산면 남리리"},
  {c:"4682040029",n:"황산면 연당리"},
  {c:"4682040030",n:"황산면 외입리"},
  {c:"4682040031",n:"황산면 부곡리"},
  {c:"4682040032",n:"황산면 관춘리"},
  {c:"4682040033",n:"황산면 옥동리"},
  {c:"4682041021",n:"산이면 노송리"},
  {c:"4682041022",n:"산이면 금송리"},
  {c:"4682041023",n:"산이면 덕호리"},
  {c:"4682041024",n:"산이면 예정리"},
  {c:"4682041025",n:"산이면 송천리"},
  {c:"4682041026",n:"산이면 초송리"},
  {c:"4682041027",n:"산이면 진산리"},
  {c:"4682041028",n:"산이면 대진리"},
  {c:"4682041029",n:"산이면 덕송리"},
  {c:"4682041030",n:"산이면 구성리"},
  {c:"4682041031",n:"산이면 상공리"},
  {c:"4682041032",n:"산이면 부동리"},
  {c:"4682041033",n:"산이면 금호리"},
  {c:"4682042021",n:"문내면 용암리"},
  {c:"4682042022",n:"문내면 석교리"},
  {c:"4682042023",n:"문내면 동외리"},
  {c:"4682042024",n:"문내면 선두리"},
  {c:"4682042025",n:"문내면 학동리"},
  {c:"4682042026",n:"문내면 서상리"},
  {c:"4682042027",n:"문내면 예락리"},
  {c:"4682042028",n:"문내면 난대리"},
  {c:"4682042029",n:"문내면 충평리"},
  {c:"4682042030",n:"문내면 무고리"},
  {c:"4682042031",n:"문내면 고당리"},
  {c:"4682042032",n:"문내면 고평리"},
  {c:"4682043021",n:"화원면 청용리"},
  {c:"4682043022",n:"화원면 금평리"},
  {c:"4682043023",n:"화원면 신덕리"},
  {c:"4682043024",n:"화원면 영호리"},
  {c:"4682043025",n:"화원면 마산리"},
  {c:"4682043026",n:"화원면 구림리"},
  {c:"4682043027",n:"화원면 월호리"},
  {c:"4682043028",n:"화원면 매월리"},
  {c:"4682043029",n:"화원면 후산리"},
  {c:"4682043030",n:"화원면 인지리"},
  {c:"4682043031",n:"화원면 주광리"},
  {c:"4682043032",n:"화원면 화봉리"},
  {c:"4682043033",n:"화원면 산호리"},
  {c:"4682043034",n:"화원면 장춘리"},
  {c:"4682043035",n:"화원면 성산리"},
  {c:"4682043036",n:"화원면 치하리"}
];

const LAND_CODES = ["01","02","03","04","06"];
const LAND_NAMES = {"01":"논","02":"밭","03":"과수","04":"시설","06":"비경지"};

// EPSG:5179 (UTM-K) 좌표계 정의
proj4.defs("EPSG:5179",
  "+proj=tmerc +lat_0=38 +lon_0=127.5 +k=0.9996 +x_0=1000000 +y_0=2000000 " +
  "+ellps=GRS80 +towgs84=0,0,0,0,0,0,0 +units=m +no_defs");

// ============================================================
// 2. 상태
// ============================================================
let allParcels = [];        // 수집된 필지 (uid 기준 dedupe)
const seenUids = new Set();
let stopped = false;
let running = false;
let startTime = null;
const stats = { done: 0, total: 0, fail: 0 };
const perEupmyeon = {};     // 읍면별 카운트

// ============================================================
// 3. 로그
// ============================================================
function log(msg, type) {
  const el = document.getElementById("log");
  const time = new Date().toTimeString().slice(0,8);
  const line = document.createElement("div");
  if (type) line.className = type;
  line.textContent = `[${time}] ${msg}`;
  el.appendChild(line);
  el.scrollTop = el.scrollHeight;
}

// ============================================================
// 4. 진행률 업데이트
// ============================================================
function updateStats() {
  document.getElementById("stat-done").textContent = stats.done;
  document.getElementById("stat-total").textContent = stats.total;
  document.getElementById("stat-parcels").textContent = allParcels.length;
  document.getElementById("stat-fail").textContent = stats.fail;
  const pct = stats.total > 0 ? (stats.done / stats.total * 100) : 0;
  document.getElementById("progress-fill").style.width = pct + "%";
  document.getElementById("progress-text").textContent = pct.toFixed(1) + "% (" + stats.done + "/" + stats.total + ")";
  if (startTime) {
    const elapsed = Math.floor((Date.now() - startTime) / 1000);
    const m = Math.floor(elapsed/60), s = elapsed%60;
    document.getElementById("stat-time").textContent = m + "분 " + s + "초";
  }
}

// ============================================================
// 5. 단일 API 호출 (Promise)
// ============================================================
function callAPI(bjdCd, landCd) {
  return new Promise((resolve) => {
    $.ajax({
      url: API_URL,
      type: "GET",
      dataType: "jsonp",
      jsonpCallback: "searchCallback",
      timeout: 30000,
      data: {
        apiKey: API_KEY,
        domain: DOMAIN,
        bjdCd: bjdCd,
        landCd: landCd,
        columnType: "ENG",
        apiVersion: "v2"
      },
      success: (resp) => resolve({ ok: true, data: resp }),
      error: (xhr, st, err) => resolve({ ok: false, error: err || st })
    });
  });
}

// ============================================================
// 6. 메인 루프
// ============================================================
async function runExtraction() {
  // 선택된 농경지 분류
  const selectedLands = LAND_CODES.filter(c => document.getElementById("opt-" + c).checked);
  if (selectedLands.length === 0) {
    alert("최소 1개 농경지 분류를 선택하세요.");
    return;
  }
  const delay = parseInt(document.getElementById("opt-delay").value) || 150;

  stats.total = CODES.length * selectedLands.length;
  stats.done = 0;
  stats.fail = 0;
  stopped = false;
  running = true;
  startTime = Date.now();

  document.getElementById("btn-start").disabled = true;
  document.getElementById("btn-stop").disabled = false;
  document.getElementById("btn-csv").disabled = true;
  document.getElementById("btn-geojson").disabled = true;

  log("추출 시작 - 총 " + stats.total + "건 호출 예정", "info");

  outer: for (const region of CODES) {
    for (const landCd of selectedLands) {
      if (stopped) break outer;

      const r = await callAPI(region.c, landCd);
      stats.done++;

      if (r.ok && r.data?.output?.farmmapData?.data) {
        const items = r.data.output.farmmapData.data;
        let added = 0;
        items.forEach(item => {
          if (!seenUids.has(item.uid)) {
            seenUids.add(item.uid);
            allParcels.push(item);
            added++;
          }
        });
        // 읍면 단위 카운트
        const eupmyeon = region.n.split(" ")[0];
        perEupmyeon[eupmyeon] = (perEupmyeon[eupmyeon] || 0) + added;

        if (items.length > 0) {
          log("✓ " + region.n + " [" + LAND_NAMES[landCd] + "] " + items.length + "건 (신규 " + added + ")");
        }
      } else if (!r.ok) {
        stats.fail++;
        log("✗ " + region.n + " [" + LAND_NAMES[landCd] + "] 실패: " + r.error, "error");
      }

      updateStats();
      await new Promise(res => setTimeout(res, delay));
    }
  }

  running = false;
  document.getElementById("btn-start").disabled = false;
  document.getElementById("btn-stop").disabled = true;
  if (allParcels.length > 0) {
    document.getElementById("btn-csv").disabled = false;
    document.getElementById("btn-geojson").disabled = false;
  }

  if (stopped) {
    log("⏸ 중지됨", "warn");
  } else {
    log("🎉 추출 완료! 총 " + allParcels.length + "건 수집 (실패 " + stats.fail + "건)", "info");
  }

  // 읍면별 요약 표시
  showSummary();
}

// ============================================================
// 7. 읍면별 요약
// ============================================================
function showSummary() {
  if (Object.keys(perEupmyeon).length === 0) return;
  const tbl = document.getElementById("summary-table");
  let html = "<thead><tr><th>읍면</th><th>필지 수</th></tr></thead><tbody>";
  Object.keys(perEupmyeon).sort().forEach(k => {
    html += "<tr><td>" + k + "</td><td>" + perEupmyeon[k].toLocaleString() + "</td></tr>";
  });
  html += "<tr style='font-weight:bold;background:#fff3cd;'><td>합계</td><td>" + allParcels.length.toLocaleString() + "</td></tr>";
  html += "</tbody>";
  tbl.innerHTML = html;
  document.getElementById("summary-panel").style.display = "block";
}

// ============================================================
// 8. CSV 다운로드
// ============================================================
function downloadCSV() {
  const headers = ["uid","pnu","stdg_cd","stdg_addr","clsf_nm","ldcg_cd","area_m2","is_farming","source_nm","flight_ymd","updt_ymd","cad_con_ra"];
  const rows = [headers.join(",")];

  allParcels.forEach(p => {
    const isFarming = (p.clsf_nm && p.clsf_nm !== "비경지") ? "Y" : "N";
    const row = [
      p.uid || "",
      p.pnu || "",
      p.stdg_cd || "",
      '"' + (p.stdg_addr || "").replace(/"/g,'""') + '"',
      p.clsf_nm || "",
      p.ldcg_cd || "",
      p.area || "",
      isFarming,
      '"' + (p.source_nm || "").replace(/"/g,'""') + '"',
      p.flight_ymd || "",
      p.updt_ymd || "",
      p.cad_con_ra || ""
    ];
    rows.push(row.join(","));
  });

  const csv = String.fromCharCode(0xFEFF) + rows.join("\n");
  saveFile(csv, "haenam_farmmap_" + ts() + ".csv", "text/csv;charset=utf-8");
}

// ============================================================
// 9. GeoJSON 다운로드 (EPSG:5179 → EPSG:4326 변환)
// ============================================================
function downloadGeoJSON() {
  const features = [];
  let skipped = 0;

  allParcels.forEach(p => {
    if (!p.geometry || !p.geometry[0] || !p.geometry[0].xy) {
      skipped++;
      return;
    }
    const xy = p.geometry[0].xy;
    const ring = xy.map(pt => proj4("EPSG:5179", "EPSG:4326", [pt.x, pt.y]));

    // 폐곡선 보장
    if (ring.length > 0) {
      const first = ring[0], last = ring[ring.length-1];
      if (first[0] !== last[0] || first[1] !== last[1]) {
        ring.push([first[0], first[1]]);
      }
    }

    features.push({
      type: "Feature",
      properties: {
        uid: p.uid,
        pnu: p.pnu,
        stdg_cd: p.stdg_cd,
        stdg_addr: p.stdg_addr || "",
        clsf_nm: p.clsf_nm || "",
        ldcg_cd: p.ldcg_cd || "",
        area_m2: parseFloat(p.area || 0),
        is_farming: (p.clsf_nm && p.clsf_nm !== "비경지"),
        source_nm: p.source_nm || "",
        flight_ymd: p.flight_ymd || ""
      },
      geometry: {
        type: "Polygon",
        coordinates: [ring]
      }
    });
  });

  if (skipped > 0) log("⚠ geometry 없음 " + skipped + "건 제외", "warn");

  const geojson = {
    type: "FeatureCollection",
    crs: { type: "name", properties: { name: "urn:ogc:def:crs:OGC:1.3:CRS84" } },
    features: features
  };

  saveFile(JSON.stringify(geojson), "haenam_farmmap_" + ts() + ".geojson", "application/geo+json");
}

// ============================================================
// 10. 유틸
// ============================================================
function saveFile(content, name, mime) {
  const blob = new Blob([content], { type: mime });
  const url = URL.createObjectURL(blob);
  const a = document.createElement("a");
  a.href = url; a.download = name;
  document.body.appendChild(a); a.click(); document.body.removeChild(a);
  URL.revokeObjectURL(url);
  log("💾 " + name + " 저장됨", "info");
}

function ts() {
  const d = new Date();
  return d.getFullYear() + ("0"+(d.getMonth()+1)).slice(-2) + ("0"+d.getDate()).slice(-2) + "_" + ("0"+d.getHours()).slice(-2) + ("0"+d.getMinutes()).slice(-2);
}

// ============================================================
// 11. 버튼 이벤트
// ============================================================
document.getElementById("btn-start").addEventListener("click", runExtraction);
document.getElementById("btn-stop").addEventListener("click", () => { stopped = true; });
document.getElementById("btn-csv").addEventListener("click", downloadCSV);
document.getElementById("btn-geojson").addEventListener("click", downloadGeoJSON);
document.getElementById("btn-clear").addEventListener("click", () => {
  if (running) { alert("실행 중에는 초기화할 수 없습니다."); return; }
  if (allParcels.length > 0 && !confirm("수집된 " + allParcels.length + "건을 모두 지웁니다. 계속할까요?")) return;
  allParcels = []; seenUids.clear();
  Object.keys(perEupmyeon).forEach(k => delete perEupmyeon[k]);
  stats.done = 0; stats.total = 0; stats.fail = 0; startTime = null;
  document.getElementById("stat-time").textContent = "-";
  document.getElementById("progress-fill").style.width = "0%";
  document.getElementById("progress-text").textContent = "대기 중";
  document.getElementById("btn-csv").disabled = true;
  document.getElementById("btn-geojson").disabled = true;
  document.getElementById("log").innerHTML = "";
  document.getElementById("summary-panel").style.display = "none";
  updateStats();
  log("초기화 완료", "info");
});

// 시작 시 안내
log("준비 완료. 178개 리 × 농경지 분류 만큼 호출합니다.", "info");
log("기본 선택: 논, 밭, 과수, 시설 (4종) → 약 712건 호출, 약 2분 소요 예상", "info");
log("호출 간격이 짧으면 서버가 차단할 수 있으니 150ms 이상 권장", "warn");
</script>
</body>
</html>